# HW3 - Seeded Deutsch-Jozsa Oracle Classification

In this assignment, you will implement a personalized Deutsch-Jozsa circuit.
Your seed generates an oracle of the form:

`f(x) = mask · x XOR output_offset`

where `mask` is given in `q[0], q[1], ..., q[n-1]` order.

Your job is to:
1. classify the oracle as **constant** or **balanced** before building the circuit,
2. build the seeded oracle,
3. run the Deutsch-Jozsa circuit on the ideal simulator,
4. interpret the displayed bitstring using the measurement map,
5. optionally run a smaller version on IBM hardware,
6. export `answers.json`.

Do not hard-code an answer. Your output must come from your own seeded circuit execution.

In [ ]:
%pip -q install qiskit qiskit-aer matplotlib jsonschema

In [ ]:
import json, math, hashlib
from typing import List, Dict, Any
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator

## 1. Student ID and seed

Replace `demo_student` with your assigned student ID. Do not change it after generating results.

In [ ]:
STUDENT_ID = "demo_student"
ASSIGNMENT_ID = "HW3"
SHOTS = 2048
HARDWARE_SHOTS = 4096

## 2. Generate your seeded oracle configuration

This function gives you the assignment configuration. It gives you the mask and measurement map, but it does not give you the expected measured result.

Important: `mask_q0_to_qn` is listed in `q[0]...q[n-1]` order.

In [ ]:
def stable_seed(student_id: str, assignment_id: str = "HW3") -> int:
    key = f"{assignment_id}|{student_id}".encode("utf-8")
    return int(hashlib.sha256(key).hexdigest()[:12], 16)


def bits_from_seed(seed: int, n: int, offset: int = 0):
    return [(seed >> (offset + i)) & 1 for i in range(n)]


def is_palindrome(bits):
    return bits == list(reversed(bits))


def make_nonzero_nonpal_mask(seed: int, n: int):
    for shift in range(0, 40, 3):
        mask = bits_from_seed(seed, n, shift)
        if any(mask) and not is_palindrome(mask):
            return mask
    mask = [0] * n
    mask[0] = 1
    mask[-2] = 1
    if is_palindrome(mask):
        mask[1] ^= 1
    return mask


def generate_config(student_id: str, assignment_id: str = "HW3"):
    seed = stable_seed(student_id, assignment_id)
    n = 3 + (seed % 3)
    make_constant = ((seed >> 5) % 3 == 0)
    if make_constant:
        mask = [0] * n
    else:
        mask = make_nonzero_nonpal_mask(seed >> 8, n)
    output_offset = (seed >> 17) & 1

    measurement_mode = "reversed" if ((seed >> 19) & 1) else "direct"
    if measurement_mode == "direct":
        measurement_map = [[i, i] for i in range(n)]
    else:
        measurement_map = [[i, n - 1 - i] for i in range(n)]

    hw_n = 3
    hw_make_constant = ((seed >> 23) % 4 == 0)
    if hw_make_constant:
        hw_mask = [0] * hw_n
    else:
        hw_mask = make_nonzero_nonpal_mask(seed >> 25, hw_n)
    hw_output_offset = (seed >> 31) & 1
    hw_measurement_mode = "reversed" if ((seed >> 32) & 1) else "direct"
    if hw_measurement_mode == "direct":
        hw_measurement_map = [[i, i] for i in range(hw_n)]
    else:
        hw_measurement_map = [[i, hw_n - 1 - i] for i in range(hw_n)]

    return {
        "assignment_id": assignment_id,
        "student_id": student_id,
        "seed": seed,
        "num_input_qubits": n,
        "mask_q0_to_qn": mask,
        "output_offset": output_offset,
        "measurement_mode": measurement_mode,
        "measurement_map": measurement_map,
        "hardware_num_input_qubits": hw_n,
        "hardware_mask_q0_to_qn": hw_mask,
        "hardware_output_offset": hw_output_offset,
        "hardware_measurement_mode": hw_measurement_mode,
        "hardware_measurement_map": hw_measurement_map,
    }

config = generate_config(STUDENT_ID, ASSIGNMENT_ID)
print(json.dumps(config, indent=2))

## 3. Classify the oracle before building it

Use the mask to decide whether your oracle is constant or balanced.

Rules:
- If every mask bit is 0, then the function does not depend on the input. It is **constant**.
- If at least one mask bit is 1, then the function is a parity function over selected input bits. It is **balanced**.

`output_offset` may flip every output value, but it does not change whether the oracle is constant or balanced.

In [ ]:
# TODO: replace this with your classification logic.
# predicted_oracle_classification should be exactly "constant" or "balanced".

predicted_oracle_classification = "TODO"

print("My oracle classification:", predicted_oracle_classification)

## 4. Build the seeded oracle

For the oracle `f(x) = mask · x XOR output_offset`:
- If `output_offset == 1`, apply X to the output qubit.
- For each mask bit equal to 1, apply CX from that input qubit to the output qubit.

Do not measure inside the oracle.

In [ ]:
def build_oracle(qc, input_qubits, output_qubit, mask_q0_to_qn, output_offset):
    # TODO: implement the seeded oracle.
    # Hint 1: output_offset == 1 means apply X to output_qubit.
    # Hint 2: mask[i] == 1 means apply CX(input_qubits[i], output_qubit).

    # YOUR CODE HERE

    return qc

## 5. Build the full Deutsch-Jozsa circuit

Circuit structure:
1. Prepare the output/ancilla qubit in `|1>`.
2. Apply H to all qubits, so the output becomes `|->` and inputs become a superposition.
3. Apply the oracle.
4. Apply H to input qubits only.
5. Measure input qubits only using the seeded measurement map.

The output/ancilla qubit is not measured for the main answer.

In [ ]:
def build_deutsch_jozsa_circuit(config):
    n = config["num_input_qubits"]
    q = QuantumRegister(n + 1, "q")
    c = ClassicalRegister(n, "c")
    qc = QuantumCircuit(q, c)
    output = q[n]

    # TODO: prepare output qubit in |->.
    # Hint: X then H.

    # TODO: apply H to all input qubits.

    qc.barrier()
    build_oracle(qc, [q[i] for i in range(n)], output, config["mask_q0_to_qn"], config["output_offset"])
    qc.barrier()

    # TODO: apply H to all input qubits again.

    qc.barrier()
    # TODO: measure input qubits according to config["measurement_map"].

    return qc

qc = build_deutsch_jozsa_circuit(config)
qc.draw("mpl")

## 6. Predict the expected bitstring

Do this before looking at simulator counts.

Rules:
- If the oracle is constant, the input-register result should be all zeros.
- If the oracle is balanced, the logical input-register result should match the mask.
- The logical string is written as `q[n-1]...q[0]`.
- The displayed Qiskit count string is written as `c[n-1]...c[0]`, using your measurement map.

In [ ]:
# TODO: compute these values yourself.
# expected_logical should be the expected input-register result in q[n-1]...q[0] order.
# expected_display should be the expected displayed count string in c[n-1]...c[0] order.

expected_logical = "TODO"
expected_display = "TODO"

print("Expected logical qubit string q[n-1]...q[0]:", expected_logical)
print("Expected displayed count string c[n-1]...c[0]:", expected_display)

## 7. Run the ideal simulator

The simulator should be strongly dominated by the expected displayed bitstring. For this exact Deutsch-Jozsa construction, it should usually be all shots on one bitstring.

In [ ]:
sim = AerSimulator(seed_simulator=config["seed"] % (2**32 - 1))
tqc = transpile(qc, sim, seed_transpiler=config["seed"] % (2**32 - 1))
result = sim.run(tqc, shots=SHOTS).result()
simulator_counts = {str(k): int(v) for k, v in result.get_counts().items()}
dominant_bitstring = max(simulator_counts, key=simulator_counts.get)

print("Simulator counts:", simulator_counts)
print("Dominant bitstring:", dominant_bitstring)
print("Expected displayed bitstring:", expected_display)
print("Match?", dominant_bitstring == expected_display)
plot_histogram(simulator_counts, title="HW3 Deutsch-Jozsa simulator counts")

## 8. Record circuit metrics

These metrics help detect missing gates, accidental shortcut circuits, and transpiler-related changes.

In [ ]:
original_depth = qc.depth()
transpiled_depth_simulator = tqc.depth()
operation_counts_simulator = {str(k): int(v) for k, v in tqc.count_ops().items()}

print("Original depth:", original_depth)
print("Transpiled simulator depth:", transpiled_depth_simulator)
print("Operation counts:", operation_counts_simulator)

## 9. Optional IBM hardware extension

This section is optional/research-oriented. Run it only if you have IBM Quantum access.

The hardware task uses a smaller 3-input-qubit seeded Deutsch-Jozsa circuit to reduce queue time and noise.

In [ ]:
# Optional. Uncomment and run if you have IBM Quantum access.
# %pip -q install qiskit-ibm-runtime

In [ ]:
# Optional IBM hardware code. Leave hardware_run_completed = False if not running hardware.

hardware_run_completed = False
hardware_backend = None
hardware_job_id = None
hardware_counts = {}
hardware_dominant_bitstring = None
hardware_expected_percentage = None
hardware_result_classification = None

# If you run hardware, adapt this cell with your IBM token/account setup.
# from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
# from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
#
# def build_hardware_circuit(config):
#     hw_config = dict(config)
#     hw_config["num_input_qubits"] = config["hardware_num_input_qubits"]
#     hw_config["mask_q0_to_qn"] = config["hardware_mask_q0_to_qn"]
#     hw_config["output_offset"] = config["hardware_output_offset"]
#     hw_config["measurement_map"] = config["hardware_measurement_map"]
#     return build_deutsch_jozsa_circuit(hw_config)
#
# service = QiskitRuntimeService(channel="ibm_quantum")
# backend = service.least_busy(operational=True, simulator=False, min_num_qubits=4)
# hardware_backend = backend.name
#
# hw_qc = build_hardware_circuit(config)
# pm = generate_preset_pass_manager(backend=backend, optimization_level=1, seed_transpiler=config["seed"] % (2**32 - 1))
# isa_circuit = pm.run(hw_qc)
#
# sampler = Sampler(backend)
# job = sampler.run([isa_circuit], shots=HARDWARE_SHOTS)
# hardware_job_id = job.job_id()
# print("Backend:", hardware_backend)
# print("Job ID:", hardware_job_id)
# print("Status:", job.status())
#
# result = job.result()
# pub_result = result[0]
# try:
#     hardware_counts = pub_result.data.c.get_counts()
# except Exception:
#     hardware_counts = None
#     for field_name in dir(pub_result.data):
#         if field_name.startswith("_"):
#             continue
#         field = getattr(pub_result.data, field_name)
#         if hasattr(field, "get_counts"):
#             hardware_counts = field.get_counts()
#             print("Counts extracted from classical register:", field_name)
#             break
#     if hardware_counts is None:
#         raise RuntimeError("Could not extract counts from SamplerV2 result.")
#
# hardware_counts = {str(k): int(v) for k, v in hardware_counts.items()}
# hardware_dominant_bitstring = max(hardware_counts, key=hardware_counts.get)
# hardware_expected = "TODO: compute expected hardware display string"
# hardware_expected_percentage = 100 * hardware_counts.get(hardware_expected, 0) / HARDWARE_SHOTS
# hardware_run_completed = True
#
# if hardware_expected_percentage >= 70:
#     hardware_result_classification = "successful"
# elif hardware_expected_percentage >= 40:
#     hardware_result_classification = "partial"
# elif hardware_dominant_bitstring == hardware_expected:
#     hardware_result_classification = "inconclusive"
# else:
#     hardware_result_classification = "failed"
#
# print("Hardware counts:", hardware_counts)
# print("Dominant:", hardware_dominant_bitstring)
# print("Expected percentage:", hardware_expected_percentage)
# print("Hardware result classification:", hardware_result_classification)
# plot_histogram(hardware_counts, title=f"IBM hardware Deutsch-Jozsa on {hardware_backend}")

## 10. Reflection questions

Answer each in 2-4 sentences.

**Q1. Oracle classification:** How did you classify your oracle as constant or balanced before building it? How did the mask and output offset affect that decision?

**Q2. Bit ordering:** How did the measurement map affect the displayed Qiskit count string? Why can `c[n-1]...c[0]` differ from the mask written in `q[0]...q[n-1]` order?

**Q3. Hardware noise:** If you ran hardware, compare your hardware counts to the simulator counts. If you did not run hardware, explain what noise effects you would expect and why.

In [ ]:
reflection_oracle_classification = "TODO: answer Q1 here"
reflection_bit_order = "TODO: answer Q2 here"
reflection_hardware_noise = "TODO: answer Q3 here"

## 11. Export answers.json

Submit this file. The hidden autograder will regenerate your reference output from your student ID and compare it with your JSON.

In [ ]:
answers = {
    "assignment_id": ASSIGNMENT_ID,
    "student_id": STUDENT_ID,
    "seed": config["seed"],
    "num_input_qubits": config["num_input_qubits"],
    "shots": SHOTS,
    "mask_q0_to_qn": config["mask_q0_to_qn"],
    "output_offset": config["output_offset"],
    "measurement_mode": config["measurement_mode"],
    "measurement_map": config["measurement_map"],
    "predicted_oracle_classification": predicted_oracle_classification,
    "expected_logical_qn_to_q0": expected_logical,
    "expected_display_bitstring": expected_display,
    "simulator_counts": simulator_counts,
    "dominant_bitstring": dominant_bitstring,
    "original_depth": original_depth,
    "transpiled_depth_simulator": transpiled_depth_simulator,
    "operation_counts_simulator": operation_counts_simulator,
    "hardware_run_completed": hardware_run_completed,
    "hardware_backend": hardware_backend,
    "hardware_job_id": hardware_job_id,
    "hardware_counts": hardware_counts,
    "hardware_dominant_bitstring": hardware_dominant_bitstring,
    "hardware_expected_percentage": hardware_expected_percentage,
    "hardware_result_classification": hardware_result_classification,
    "reflection_oracle_classification": reflection_oracle_classification,
    "reflection_bit_order": reflection_bit_order,
    "reflection_hardware_noise": reflection_hardware_noise,
}

with open("answers.json", "w", encoding="utf-8") as f:
    json.dump(answers, f, indent=2)

print(json.dumps(answers, indent=2))
print("Saved answers.json")